In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import functools as ft
from IPython.display import display


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp   = path_users / 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
path_main = path_sp / 'Data' 
path_in = path_main / 'Vibrant and Inclusive Places' / 'People and Community'/ 'Pop and Demographics'
path_prod = path_sp / 'Products'
path_sdr = path_prod / 'Small Data Requests' / '2025' / 'Yuba-Sutter Transit Authority'

path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'DOF'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'



In [ ]:


file_in = path_in / 'DOF_E5_and_E8_Jurisdictions.xlsx'
df = pd.read_excel(file_in)

counties = ['Yuba', 'Sutter']

df = df[df['County'].isin(counties)]
df = df.reset_index(drop=True)

df = df[['Year', 'County', 'Jurisdiction', 'Housing Units', 'Population']]

df.loc[df['Jurisdiction'] == 'Unincorporated', 'Jurisdiction'] = df['Jurisdiction'] + ' ' + df['County'] + ' County'

df['County Pop'] = df.groupby(['Year', 'County'])['Population'].transform('sum')
df['Percent of County Population'] = df['Population']/df['County Pop']
df = df[['Year', 'County', 'Jurisdiction', 'Housing Units', 'Population', 'Percent of County Population']]

df_counties = df.copy()
df_counties = df_counties.groupby(['Year', 'County'], as_index=False).sum()
df_counties['Jurisdiction'] = 'Total ' + df_counties['County'] + ' County'

df = pd.concat([df, df_counties])

df['Sort'] = pd.Categorical(df['Jurisdiction'], [
    'Live Oak'
    , 'Yuba'
    , 'Unincorporated Sutter County'
    , 'Total Sutter County'
    , 'Marysville'
    , 'Wheatland'
    , 'Unincorporated Yuba County'
    , 'Total Yuba County'
])

df = df.sort_values(['Year', 'Sort'], ascending=[False, True])
df = df.drop('Sort', axis=1)

df



In [ ]:
file_sdr = path_sdr / f'Yuba Sutter Transit_all years.xlsx'
df.to_excel(file_sdr, index=False)